# Session 2: the numbers inside the boxes
**A. Real-world problem.** A liquid mixture of benzene and toluene is heated until it
just starts to boil, then a little more. Part of it turns to vapour, which is richer in the
more volatile benzene. At what temperature does it start to boil, and how much vapour do we
get at a chosen temperature? A flowsheet program answers this thousands of times inside its
boxes; today you write that box.

Symbols: n_i = molar flow of component i (mol/s); z_i, x_i, y_i = mole fractions in the
feed, liquid and vapour; P = pressure (kPa); T = temperature (K); V = vapour fraction
(moles of vapour per mole of feed).

In [ ]:
import pathlib
from dataclasses import dataclass

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

NAMES = ["benzene", "toluene", "ethanol"]
# Antoine: log10(P/mmHg) = A - B / (C + T/degC). Rounded textbook values, valid roughly 10-100 degC.
ANTOINE = np.array([[6.90565, 1211.033, 220.790],
                    [6.95464, 1344.800, 219.482],
                    [8.20417, 1642.890, 230.300]])
CP = np.array([136.0, 157.0, 112.0])          # liquid heat capacity, J/(mol K), approximate
DHVAP = np.array([30800.0, 33200.0, 38600.0])  # heat of vaporisation, J/mol, approximate
T_REF = 298.15                                 # enthalpy reference, K
P_ATM = 101.325                                # kPa
MMHG_TO_KPA = 0.1333224

## Step 1: a stream object
A stream is "what flows in a pipe": molar flow of each component, a temperature, a
pressure and a phase tag ("L" liquid or "V" vapour). Enthalpy uses a constant Cp:
H = sum n_i * (Cp_i * (T - T_ref)), plus n_i * dHvap_i if the stream is vapour.

In [ ]:
@dataclass
class Stream:
    n: np.ndarray          # component molar flows, mol/s
    T: float               # K
    P: float               # kPa
    phase: str = "L"

    def total(self):
        return float(np.sum(self.n))

    def z(self):
        return self.n / self.total()

    def enthalpy(self):
        """Enthalpy flow in W (J/s) relative to the pure liquids at T_REF."""
        h = CP * (self.T - T_REF) + (DHVAP if self.phase == "V" else 0.0)
        return float(np.sum(self.n * h))

## Step 2: unit blocks are functions
Each block takes streams in and returns streams out (and a heat or work duty, in W).

In [ ]:
def mixer(streams):
    """Adiabatic mixer of liquid streams; outlet pressure = lowest inlet pressure."""
    if any(s.phase != "L" for s in streams):
        raise ValueError("mixer: this teaching model mixes liquid streams only")
    n = sum(s.n for s in streams)
    hsum = sum(float(np.sum(s.n * CP)) * (s.T - T_REF) for s in streams)
    T = T_REF + hsum / float(np.sum(n * CP))
    return Stream(n, T, min(s.P for s in streams), "L")


def splitter(s, fractions):
    """Split one stream into several of identical composition. Fractions must sum to 1."""
    if abs(sum(fractions) - 1.0) > 1e-12 or min(fractions) < 0:
        raise ValueError("splitter: fractions must be >= 0 and sum to 1")
    return [Stream(f * s.n, s.T, s.P, s.phase) for f in fractions]


def heater(s, T_out):
    """Change temperature at constant pressure. Returns (stream, duty in W); negative duty = cooling."""
    out = Stream(s.n.copy(), T_out, s.P, s.phase)
    return out, out.enthalpy() - s.enthalpy()


def pump(s, P_out, eta=0.7, vm=1.0e-4):
    """Raise the pressure of a liquid. vm = molar volume (m3/mol, about 1e-4 for these liquids).
    Returns (stream, shaft power in W). Temperature rise is ignored."""
    w = s.total() * vm * (P_out - s.P) * 1000.0 / eta
    return Stream(s.n.copy(), s.T, P_out, s.phase), w


def valve(s, P_out):
    """Pressure drop with no work and no heat; a liquid keeps its temperature in this model."""
    if P_out > s.P:
        raise ValueError("valve: outlet pressure must not exceed inlet pressure")
    return Stream(s.n.copy(), s.T, P_out, s.phase)

## Step 3: vapour pressure and Raoult's law
The Antoine equation gives the vapour pressure P_sat(T) of a pure liquid. For an ideal
mixture Raoult's law says the partial pressure of i is x_i * P_sat_i(T), so the
K-value (K_i = y_i / x_i) is K_i = P_sat_i / P. At the bubble point the liquid first
boils: sum x_i P_sat_i(T) = P. At the dew point the vapour first condenses: sum y_i / K_i = 1.

In [ ]:
def psat(T, idx=None):
    """Vapour pressure in kPa at T (K). All components if idx is None."""
    a = ANTOINE if idx is None else ANTOINE[idx]
    return MMHG_TO_KPA * 10.0 ** (a[..., 0] - a[..., 1] / (a[..., 2] + T - 273.15))


def k_values(T, P):
    return psat(T) / P


def _bisect(f, lo, hi, tol=1e-10, maxit=200):
    """Root of f in [lo, hi] by bisection; returns (root, iterations). f(lo) and f(hi) must differ in sign."""
    flo = f(lo)
    for it in range(1, maxit + 1):
        mid = 0.5 * (lo + hi)
        fm = f(mid)
        if flo * fm <= 0:
            hi = mid
        else:
            lo, flo = mid, fm
        if hi - lo < tol:
            break
    return 0.5 * (lo + hi), it


def bubble_T(x, P):
    """Bubble-point temperature (K) of liquid x at pressure P (kPa), ideal. Returns (T, y)."""
    x = np.asarray(x, dtype=float)
    mask = x > 0
    f = lambda T: float(np.sum(x[mask] * psat(T)[mask])) / P - 1.0
    T, _ = _bisect(f, 200.0, 500.0)
    y = x * k_values(T, P)
    return T, y / y.sum()


def dew_T(y, P):
    """Dew-point temperature (K) of vapour y at pressure P (kPa), ideal. Returns (T, x)."""
    y = np.asarray(y, dtype=float)
    mask = y > 0
    f = lambda T: float(np.sum(y[mask] / k_values(T, P)[mask])) - 1.0
    T, _ = _bisect(f, 200.0, 500.0)
    x = y / k_values(T, P)
    return T, x / x.sum()

## Step 4: the Rachford-Rice flash
Feed z (1 mole), split into vapour V and liquid 1 - V. Component balance z_i = V y_i + (1-V) x_i
with y_i = K_i x_i gives x_i = z_i / (1 + V (K_i - 1)). Because the mole fractions of the
liquid must sum to 1 as well as those of the vapour, we need

    f(V) = sum_i z_i (K_i - 1) / (1 + V (K_i - 1)) = 0.

f falls as V grows, so there is one root between 0 and 1 (if the feed is two-phase).
f(0) <= 0 means the feed is liquid below its bubble point (V = 0). f(1) >= 0 means it is all vapour.

In [ ]:
def rr_f(V, z, K):
    return float(np.sum(z * (K - 1.0) / (1.0 + V * (K - 1.0))))


def rr_bisection(z, K, tol=1e-12):
    """Vapour fraction by bisection. Returns (V, iterations). V is clipped to 0 or 1 outside the two-phase region."""
    if rr_f(0.0, z, K) <= 0:
        return 0.0, 0
    if rr_f(1.0, z, K) >= 0:
        return 1.0, 0
    lo, hi = 0.0, 1.0
    for it in range(1, 200):
        mid = 0.5 * (lo + hi)
        if rr_f(mid, z, K) > 0:
            lo = mid
        else:
            hi = mid
        if hi - lo < tol:
            break
    return 0.5 * (lo + hi), it


def rr_newton(z, K, tol=1e-12):
    """Vapour fraction by Newton's method with a bisection safety net. Returns (V, iterations)."""
    if rr_f(0.0, z, K) <= 0:
        return 0.0, 0
    if rr_f(1.0, z, K) >= 0:
        return 1.0, 0
    lo, hi, V = 0.0, 1.0, 0.5
    for it in range(1, 100):
        f = rr_f(V, z, K)
        if f > 0:
            lo = V
        else:
            hi = V
        df = -float(np.sum(z * (K - 1.0) ** 2 / (1.0 + V * (K - 1.0)) ** 2))
        Vn = V - f / df
        if not (lo < Vn < hi):
            Vn = 0.5 * (lo + hi)
        if abs(Vn - V) < tol:
            return Vn, it
        V = Vn
    return V, it


def flash_TP(feed, T, P):
    """Isothermal flash of a stream at T, P. Returns (vapour stream, liquid stream, V, duty W).
    Duty = enthalpy out - enthalpy in (what the heater inside the box must supply)."""
    z = feed.z()
    K = k_values(T, P)
    V, _ = rr_newton(z, K)
    x = z / (1.0 + V * (K - 1.0))
    y = K * x
    x, y = x / x.sum(), y / y.sum()
    F = feed.total()
    vap = Stream(V * F * y, T, P, "V") if V > 0 else Stream(np.zeros(3), T, P, "V")
    liq = Stream((1.0 - V) * F * x, T, P, "L") if V < 1 else Stream(np.zeros(3), T, P, "L")
    return vap, liq, V, vap.enthalpy() + liq.enthalpy() - feed.enthalpy()

## Step 5: a design problem (inverse)
Instead of "given T find V", ask "given V find T": the flash temperature for 50 percent
vapour. V rises with T, so bisection on T works.

In [ ]:
def flash_T_for_vapour_fraction(z, P, V_target, lo=300.0, hi=450.0):
    """Temperature (K) at which the flash of feed z gives vapour fraction V_target (0 < V_target < 1)."""
    z = np.asarray(z, dtype=float)
    f = lambda T: rr_newton(z, k_values(T, P))[0] - V_target
    T, _ = _bisect(f, lo, hi)
    return T

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    z = np.array([0.5, 0.5, 0.0])
    Tb, yb = bubble_T(z, P_ATM)
    Td, xd = dew_T(z, P_ATM)
    print(f"equimolar benzene-toluene at 1 atm: bubble {Tb:.2f} K, dew {Td:.2f} K")
    print(f"first vapour y = {yb[:2].round(3)}")
    Th = flash_T_for_vapour_fraction(z, P_ATM, 0.5)
    print(f"flash T for 50 percent vapour: {Th:.2f} K")

    # figure 1: T-x-y diagram of benzene-toluene at 1 atm
    xb = np.linspace(0, 1, 101)
    bub = [bubble_T(np.array([x, 1 - x, 0.0]), P_ATM)[0] for x in xb]
    dew = [dew_T(np.array([x, 1 - x, 0.0]), P_ATM)[0] for x in xb]
    fig, ax = plt.subplots(figsize=(6.4, 4.2))
    ax.plot(xb, bub, label="bubble curve (liquid x)")
    ax.plot(xb, dew, label="dew curve (vapour y)")
    ax.plot([0.5], [Tb], "ko")
    ax.plot([0.5], [Td], "ks")
    ax.set_xlabel("benzene mole fraction (liquid x or vapour y)")
    ax.set_ylabel("temperature T (K)")
    ax.set_title("Benzene-toluene at 1 atm (Raoult's law)")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(OUT / "txy.png", dpi=120)

    # figure 2: vapour fraction against flash temperature
    Ts = np.linspace(Tb - 3, Td + 3, 200)
    Vs = [rr_newton(z, k_values(T, P_ATM))[0] for T in Ts]
    fig2, ax2 = plt.subplots(figsize=(6.4, 4.2))
    ax2.plot(Ts, Vs)
    ax2.axhline(0.5, color="gray", ls=":")
    ax2.plot([Th], [0.5], "ro")
    ax2.set_xlabel("flash temperature T (K)")
    ax2.set_ylabel("vapour fraction V (mol vapour / mol feed)")
    ax2.set_title("Flash curve of an equimolar feed at 1 atm")
    ax2.grid(alpha=0.3)
    fig2.tight_layout()
    fig2.savefig(OUT / "flash_curve.png", dpi=120)

## Where this stops
Real liquids are not ideal (Session 3 shows an ethanol-benzene mixture where Raoult's law
fails), and the flash here is a single box. Session 3 joins boxes into a flowsheet.

## Coding tasks
1. **Easy.** Use `bubble_T` to compute the bubble point of an equimolar benzene-toluene mixture at 2 atm (202.65 kPa) and compare with 1 atm. Predict the direction first.
2. **Stretch.** Write a function `flash_T_for_purity(z, P, y_target)` that finds the flash temperature at which the vapour reaches a benzene fraction y_target, using the same bisection helper.
3. **Explain.** Explain, using the T-x-y diagram, why no single flash of an equimolar benzene-toluene feed can produce a vapour of 0.8 benzene.

## Assessment
- **A. Conceptual.** Why can a mixture not boil at a single temperature, as a pure liquid does?
- **B. Numerical.** (a) How much heat does heating 100 mol/s of the equimolar liquid from 300 K to 350 K take? (b) What shaft power does a pump with 70 percent efficiency need to raise it by 400 kPa (molar volume 1e-4 m3/mol)? (c) What is the temperature after mixing 60 mol/s benzene at 300 K with 40 mol/s toluene at 340 K?
- **C. Graph and simulation interpretation.** From the flash curve and T-x-y diagram: what is the highest benzene fraction any single flash of the equimolar feed can give in the vapour, and what temperature gives 50 percent vapour?
- **D. Design and optimisation challenge.** Design: a flash of the equimolar feed at 1 atm must vaporise at least 30 percent of the feed and give a vapour with at least 0.65 benzene. Find the temperature window and say what a control engineer would worry about.

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    P = L.P_ATM
    z = np.array([0.5, 0.5, 0.0])
    Tb, yb = L.bubble_T(z, P)
    Td, xd = L.dew_T(z, P)
    checks.append((f"equimolar benzene-toluene at 1 atm: bubble point near 365 K (got {Tb:.1f})", abs(Tb - 365.0) < 0.5 and round(Tb, 1) == 365.3))
    checks.append((f"dew point of the equimolar mixture is 371.9 K (got {Td:.1f})", round(Td, 1) == 371.9))
    checks.append(("first vapour at the bubble point is 0.714 benzene", round(float(yb[0]), 3) == 0.714))
    pure = [round(L.bubble_T(np.eye(3)[i], P)[0], 1) for i in range(3)]
    checks.append((f"pure boiling points: benzene 353.2, toluene 383.8, ethanol 351.5 K (got {pure})", pure == [353.2, 383.8, 351.5]))
    # worked example at 368 K
    T = 368.0
    ps = L.psat(T)
    K = L.k_values(T, P)
    V, _ = L.rr_newton(z, K)
    x = z / (1.0 + V * (K - 1.0))
    y = K * x
    checks.append((f"worked example: P_sat at 368 K is 156.2 kPa (benzene) and 63.3 kPa (toluene) (got {ps[0]:.1f}, {ps[1]:.1f})", round(float(ps[0]), 1) == 156.2 and round(float(ps[1]), 1) == 63.3))
    checks.append((f"worked example: K = 1.542 and 0.6245 (got {K[0]:.3f}, {K[1]:.4f})", round(float(K[0]), 3) == 1.542 and round(float(K[1]), 4) == 0.6245))
    checks.append((f"worked example: V = 0.409, x_benzene = 0.409, y_benzene = 0.631 (got {V:.3f}, {x[0]:.3f}, {y[0]:.3f})",
                   round(V, 3) == 0.409 and round(float(x[0]), 3) == 0.409 and round(float(y[0]), 3) == 0.631))
    f_hand = 0.5 * 0.542 / (1 + 0.542 * V) + 0.5 * (-0.3755) / (1 - 0.3755 * V)
    checks.append(("hand calculation: the Rachford-Rice sum f(V) is zero at the root", abs(f_hand) < 1e-3 and abs(L.rr_f(V, z, K)) < 1e-10))
    # mass balance closure of the flash block
    feed = L.Stream(np.array([50.0, 50.0, 0.0]), 300.0, P)
    vap, liq, Vf, duty = L.flash_TP(feed, T, P)
    checks.append(("flash closes the component balance to 1e-12", float(np.max(np.abs(vap.n + liq.n - feed.n))) < 1e-12))
    checks.append(("flash energy: duty = H(vapour) + H(liquid) - H(feed) and is positive", abs(duty - (vap.enthalpy() + liq.enthalpy() - feed.enthalpy())) < 1e-6 and duty > 0))
    checks.append(("vapour is richer in benzene than the liquid (y > x)", vap.n[0] / vap.total() > liq.n[0] / liq.total()))
    # limits V = 0 at bubble and V = 1 at dew
    Vlo = L.rr_newton(z, L.k_values(Tb - 1e-6, P))[0]
    Vbub = L.rr_newton(z, L.k_values(Tb + 1e-6, P))[0]
    Vdew = L.rr_newton(z, L.k_values(Td - 1e-6, P))[0]
    Vhi = L.rr_newton(z, L.k_values(Td + 1e-6, P))[0]
    checks.append(("limit: V = 0 just below the bubble point and tiny just above it", Vlo == 0.0 and 0.0 < Vbub < 1e-4))
    checks.append(("limit: V = 1 just above the dew point and close to 1 just below it", Vhi == 1.0 and 1 - 1e-4 < Vdew < 1.0))
    # solvers
    Vb, nb = L.rr_bisection(z, K)
    Vn, nn = L.rr_newton(z, K)
    checks.append((f"bisection needs 40 iterations, Newton 4, and agree within 1e-9 (got {nb}, {nn})", nb == 40 and nn == 4 and abs(Vb - Vn) < 1e-9))
    # design problem
    Th = L.flash_T_for_vapour_fraction(z, P, 0.5)
    Kh = L.k_values(Th, P)
    Vh, _ = L.rr_newton(z, Kh)
    xh = z / (1 + Vh * (Kh - 1))
    checks.append((f"design: 50 percent vapour at 368.6 K (got {Th:.1f}), vapour 0.611 benzene, liquid 0.389",
                   round(Th, 1) == 368.6 and abs(Vh - 0.5) < 1e-9 and round(float((Kh * xh)[0]), 3) == 0.611 and round(float(xh[0]), 3) == 0.389))
    T30 = L.flash_T_for_vapour_fraction(z, P, 0.3)
    K30 = L.k_values(T30, P)
    x30 = z / (1 + 0.3 * (K30 - 1))
    checks.append((f"design window: 30 percent vapour needs 367.2 K and then y = 0.655 (got {T30:.1f}, {(K30 * x30)[0]:.3f})",
                   round(T30, 1) == 367.2 and round(float((K30 * x30)[0]), 3) == 0.655))
    def y_of(Tx):
        Kx = L.k_values(Tx, P)
        Vx, _ = L.rr_newton(z, Kx)
        return float((Kx * (z / (1 + Vx * (Kx - 1))))[0])
    checks.append(("design window: y stays at or above 0.65 up to 367.39 K but not at 367.45 K", y_of(367.39) >= 0.65 > y_of(367.45)))
    checks.append(("no single flash of the equimolar feed can exceed y = 0.714 benzene (the first-vapour value)", all(y_of(t) <= 0.7141 for t in np.linspace(365.4, 371.8, 40))))
    # blocks
    h, q = L.heater(feed, 350.0)
    checks.append((f"heater: 100 mol/s equimolar from 300 to 350 K takes 732.5 kW (got {q / 1000:.1f})", abs(q - 732500.0) < 1e-6))
    pm, w = L.pump(feed, P + 400.0)
    checks.append((f"pump: 400 kPa rise on 100 mol/s at 70 percent efficiency needs 5.71 kW (got {w / 1000:.2f})", round(w / 1000, 2) == 5.71))
    m = L.mixer([L.Stream(np.array([60.0, 0.0, 0.0]), 300.0, P), L.Stream(np.array([0.0, 40.0, 0.0]), 340.0, P)])
    checks.append((f"mixer: 60 mol/s benzene at 300 K + 40 mol/s toluene at 340 K leaves at 317.4 K (got {m.T:.1f})", round(m.T, 1) == 317.4 and abs(m.total() - 100.0) < 1e-12))
    a, b = L.splitter(feed, [0.3, 0.7])
    checks.append(("splitter keeps composition and conserves flow", abs(a.total() - 30.0) < 1e-12 and abs(a.total() + b.total() - 100.0) < 1e-12 and np.allclose(a.z(), feed.z())))
    bad = 0
    for fn in (lambda: L.splitter(feed, [0.5, 0.6]), lambda: L.valve(feed, P + 10.0)):
        try:
            fn()
        except ValueError:
            bad += 1
    checks.append(("edge: a splitter that does not sum to 1 and a valve that raises pressure are refused", bad == 2))
    v = L.valve(feed, 50.0)
    checks.append(("valve lowers pressure, keeps temperature and flow", v.P == 50.0 and v.T == feed.T and abs(v.total() - 100.0) < 1e-12))
    # edge: feed below its bubble point and above its dew point
    cold = L.flash_TP(feed, 340.0, P)
    hot = L.flash_TP(feed, 400.0, P)
    checks.append(("edge: at 340 K all liquid, at 400 K all vapour", cold[2] == 0.0 and cold[1].total() == 100.0 and hot[2] == 1.0 and abs(hot[0].total() - 100.0) < 1e-12))
    checks.append((f"the boiling window of the equimolar mixture is 6.7 K wide (got {Td - Tb:.1f})", round(Td - Tb, 1) == 6.7))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
